# 2. Limpeza

Padroniza sentinelas e decide o que entra (ou não) no reachable set antes
do merge. Não escreve output final - isso é responsabilidade de
`07_export_final.ipynb`.


In [1]:
import json
from pathlib import Path

import pandas as pd

# Notebook lives in notebooks/etl/, so the repo root is two levels up.
REPO_ROOT = Path("../..").resolve()

ROOT = REPO_ROOT / "data/Pal/Content"
PAL = ROOT / "Pal"
ITEM_DT = PAL / "DataTable/Item/DT_ItemDataTable_Common.json"
RECIPE_DT = PAL / "DataTable/Item/DT_ItemRecipeDataTable_Common.json"
BUILDOBJECT_DT = PAL / "DataTable/MapObject/Building/DT_BuildObjectDataTable_Common.json"
BENCH_RECIPES = REPO_ROOT / "src/data/bench_recipes.json"
NAMES_DT_EN = ROOT / "L10N/en/Pal/DataTable/Text/DT_ItemNameText_Common.json"
NAMES_DT_PT_BR = ROOT / "L10N/pt-BR/Pal/DataTable/Text/DT_ItemNameText_Common.json"


def load_rows(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)[0]["Rows"]


In [2]:
items_raw = load_rows(ITEM_DT)
recipes_raw = load_rows(RECIPE_DT)
buildings_raw = load_rows(BUILDOBJECT_DT)

items_df = pd.DataFrame.from_dict(items_raw, orient="index")
recipes_df = pd.DataFrame.from_dict(recipes_raw, orient="index")
buildings_df = pd.DataFrame.from_dict(buildings_raw, orient="index")

print(f"items: {items_df.shape}, recipes: {recipes_df.shape}, buildings: {buildings_df.shape}")


items: (2466, 53), recipes: (1414, 20), buildings: (498, 32)


## Padronizar sentinela `"None"`

A string literal `"None"` vira `None` real, pra parar de precisar comparar
com string em toda checagem downstream.


In [3]:
items_clean = items_df.replace("None", None)
recipes_clean = recipes_df.replace("None", None)
buildings_clean = buildings_df.replace("None", None)


## Casing: resolvedor de id case-insensitive

Detectado no profiling (`01_profiling.ipynb`) - alguns `Material_Id`/
`Product_Id` têm casing diferente do id real na tabela de items
(ex.: `"cloth"` vs `"Cloth"`). Em vez de descartar essas referências,
resolve contra o id real.


In [4]:
items_by_lower = {item_id.lower(): item_id for item_id in items_clean.index}


def resolve_item_id(raw_id):
    # pandas' .replace("None", None) upcasts these cells to NaN (a float),
    # not Python None - "is None" or plain truthiness checks silently miss
    # it and .lower() blows up on a float. pd.isna() catches both.
    if pd.isna(raw_id):
        return None
    if raw_id in items_clean.index:
        return raw_id
    return items_by_lower.get(raw_id.lower())


material_cols = [c for c in recipes_clean.columns if c.startswith("Material") and c.endswith("_Id")]
referenced_ids = set(recipes_clean["Product_Id"].dropna()) | set(
    recipes_clean[material_cols].values.flatten()
)
referenced_ids = {rid for rid in referenced_ids if pd.notna(rid)}

unresolved = sorted(rid for rid in referenced_ids if resolve_item_id(rid) is None)
print(f"{len(unresolved)} id(s) que NÃO resolvem nem direto nem por casing (órfãos de verdade):")
unresolved


0 id(s) que NÃO resolvem nem direto nem por casing (órfãos de verdade):


[]

## Candidatos a item de teste/debug

`bLegalInGame=False` é o sinal disponível, mas não é confiável sozinho -
algumas variantes de skin/NPC legítimas também usam essa flag. A lista
abaixo é para revisão manual, não um filtro automático.


In [5]:
debug_candidates = items_clean[items_clean["bLegalInGame"] == False]
print(f"{len(debug_candidates)} candidato(s) a item de debug/teste (bLegalInGame=False):")
debug_candidates.index.tolist()[:30]


575 candidato(s) a item de debug/teste (bLegalInGame=False):


['AnimalSkin',
 'AnimalSkin2',
 'Scales',
 'Scales2',
 'Axe_Tier_03',
 'Bat_NPC',
 'Berries2',
 'CaptureRope',
 'Claws',
 'Claws2',
 'ClawsPendant',
 'ClothHat',
 'ElectronicCircuit',
 'EnergyDrink',
 'Fang',
 'Fang2',
 'FangNecklace',
 'FarmCrop_Tmp',
 'FishMeat',
 'FishMeat2',
 'Flint',
 'GrilledMeat',
 'Gunpowder',
 'Handgun_NPC',
 'LaserRifle_NPC',
 'MindControlDrug',
 'IronOre',
 'Kitsunebi_Fire',
 'LargeBullet',
 'Launcher_Meat']

In [6]:
# Decisão manual: nenhum item confirmado como "só teste" até agora -
# bLegalInGame=False sozinho não basta (ver célula acima). Preencher este
# set com ids específicos só depois de revisão item a item, comparando com
# uma fonte externa curada (ver 06_validacao_cruzada_manual.ipynb).
EXCLUDE_IDS: set[str] = set()

items_cleaned_final = items_clean.drop(index=sorted(EXCLUDE_IDS), errors="ignore")
print(f"{len(items_cleaned_final)} items após limpeza ({len(EXCLUDE_IDS)} excluído(s) manualmente)")


2466 items após limpeza (0 excluído(s) manualmente)


## Resultado


In [7]:
print(f"items_cleaned_final: {items_cleaned_final.shape}")
items_cleaned_final.head()


items_cleaned_final: (2466, 53)


,OverrideName,OverrideDescription,IconName,TypeA,TypeB,Rank,Rarity,MaxStackCount,Weight,Price,...,ShieldValue,MagicAttackValue,MagicDefenseValue,PassiveSkillName,PassiveSkillName2,PassiveSkillName3,PassiveSkillName4,WazaID,CorruptionFactor,FloatValue1
Money,NaN,NaN,Money,EPalItemTypeA::Material,EPalItemTypeB::Money,1,0,99999999,0.000,1,...,0,0,0,NaN,NaN,NaN,NaN,EPalWazaID::None,0.0,0.0
AnimalSkin,NaN,NaN,AnimalSkin,EPalItemTypeA::Material,EPalItemTypeB::MaterialMonster,1,0,9999,1.000,1,...,0,0,0,NaN,NaN,NaN,NaN,EPalWazaID::None,0.0,0.0
AnimalSkin2,NaN,NaN,AnimalSkin2,EPalItemTypeA::Material,EPalItemTypeB::MaterialMonster,1,1,9999,3.000,1,...,0,0,0,NaN,NaN,NaN,NaN,EPalWazaID::None,0.0,0.0
Arrow,NaN,NaN,Arrow,EPalItemTypeA::Ammo,EPalItemTypeB::ConsumeBullet,1,0,9999,0.025,10,...,0,0,0,NaN,NaN,NaN,NaN,EPalWazaID::None,0.0,0.0
Arrow_Poison,NaN,NaN,Arrow_Poison,EPalItemTypeA::Ammo,EPalItemTypeB::ConsumeBullet,1,0,9999,0.025,40,...,0,0,0,NaN,NaN,NaN,NaN,EPalWazaID::None,0.0,0.0
